# MedCore RCM — Corrected Data Generation Notebook

**Purpose:** generate synthetic healthcare revenue-cycle data with authorization/denial consistency checks.

## Before you run
- Use **Restart Kernel and Run All** so every table is generated in order.
- The authorization rule used here is: an approval on or before the service date is considered valid for that service.
- Denial reasons are assigned only after authorization records exist, so a claim approved on/before service is not labelled “Missing Authorization.”
- Validation assertions must all pass before export.
- CSVs are written to `MedCore_RCM_Data_Corrected/`, a separate folder. Existing notebooks and the original `MedCore_RCM_Data/` exports are not overwritten.
- All records are synthetic and intended for portfolio/learning use, not real medical or insurance decisions.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reproducible results
rng = np.random.default_rng(42)

# =========================================================
# 1. PAYER MASTER
# =========================================================

payers = pd.DataFrame([
    ["PY001", "SecureHealth Insurance", "Private"],
    ["PY002", "PrimeCare Health", "Private"],
    ["PY003", "MediSure Insurance", "Private"],
    ["PY004", "WellLife Health", "Private"],
    ["PY005", "NationalCare Health", "Government"],
    ["PY006", "United Medical Assurance", "Private"],
    ["PY007", "HealthFirst Insurance", "Private"],
    ["PY008", "CareShield Health", "Private"]
], columns=[
    "Payer_ID",
    "Payer_Name",
    "Payer_Type"
])


# =========================================================
# 2. PROVIDER MASTER
# =========================================================

locations = [
    "Chennai Central",
    "Chennai East",
    "Chennai North",
    "Chennai South",
    "Chennai West",
    "Coimbatore"
]

specialty_profiles = [
    ["Cardiology", "General Medicine"],
    ["Radiology", "General Medicine"],
    ["Orthopedics", "Emergency Medicine"],
    ["Oncology", "General Medicine"],
    ["Neurology", "General Medicine"],
    ["Gastroenterology", "General Medicine"],
    ["General Medicine", "Emergency Medicine"],
    ["Cardiology", "Radiology", "General Medicine"]
]

provider_rows = []

for i in range(1, 41):

    location = locations[(i - 1) % len(locations)]

    if i <= 24:
        provider_type = "Hospital"
    else:
        provider_type = "Specialty Center"

    profile = specialty_profiles[(i - 1) % len(specialty_profiles)]

    provider_rows.append([
        f"PR{i:03d}",
        f"MedCore Provider {i:02d}",
        location,
        provider_type,
        "|".join(profile)
    ])

providers = pd.DataFrame(
    provider_rows,
    columns=[
        "Provider_ID",
        "Provider_Name",
        "Location",
        "Provider_Type",
        "Specialty_Profile"
    ]
)


# =========================================================
# 3. PROCEDURE MASTER
# =========================================================

procedure_rows = [

    ["PC001", "ECG", "Cardiology", False, 800, 2500],
    ["PC002", "Echocardiogram", "Cardiology", False, 2500, 6000],
    ["PC003", "Cardiac Stress Test", "Cardiology", True, 3000, 7500],
    ["PC004", "Cardiac Catheterization", "Cardiology", True, 45000, 120000],

    ["PC005", "X-Ray", "Radiology", False, 500, 2000],
    ["PC006", "CT Scan", "Radiology", True, 5000, 15000],
    ["PC007", "MRI Brain", "Radiology", True, 8000, 20000],
    ["PC008", "MRI Spine", "Radiology", True, 9000, 22000],
    ["PC009", "Ultrasound", "Radiology", False, 1000, 4000],

    ["PC010", "Fracture Treatment", "Orthopedics", False, 8000, 30000],
    ["PC011", "Joint Injection", "Orthopedics", True, 3000, 10000],
    ["PC012", "Knee Replacement", "Orthopedics", True, 150000, 350000],

    ["PC013", "Chemotherapy", "Oncology", True, 20000, 100000],
    ["PC014", "Radiation Therapy", "Oncology", True, 30000, 150000],
    ["PC015", "Oncology Consultation", "Oncology", False, 1000, 4000],

    ["PC016", "EEG", "Neurology", True, 2000, 6000],
    ["PC017", "Neurology Consultation", "Neurology", False, 1000, 4000],

    ["PC018", "Endoscopy", "Gastroenterology", True, 4000, 12000],
    ["PC019", "Colonoscopy", "Gastroenterology", True, 5000, 15000],
    ["PC020", "Gastroenterology Consultation", "Gastroenterology", False, 1000, 4000],

    ["PC021", "General Consultation", "General Medicine", False, 500, 2000],
    ["PC022", "Lab Panel", "General Medicine", False, 1000, 6000],

    ["PC023", "Emergency Consultation", "Emergency Medicine", False, 1000, 5000],
    ["PC024", "Emergency Procedure", "Emergency Medicine", True, 5000, 40000]
]

procedures = pd.DataFrame(
    procedure_rows,
    columns=[
        "Procedure_Code",
        "Procedure_Name",
        "Specialty",
        "Authorization_Required",
        "Min_Claim_Amount",
        "Max_Claim_Amount"
    ]
)


# =========================================================
# 4. PATIENT MASTER
# =========================================================

number_of_patients = 15000

patients = pd.DataFrame({

    "Patient_ID": [
        f"PAT{i:05d}"
        for i in range(1, number_of_patients + 1)
    ],

    "Age": rng.integers(
        18,
        86,
        number_of_patients
    ),

    "Gender": rng.choice(
        ["Female", "Male"],
        number_of_patients,
        p=[0.51, 0.49]
    )
})


# =========================================================
# 5. BASIC VALIDATION
# =========================================================

print("========== MASTER DATA VALIDATION ==========")

print("\nPatients:")
print(f"Rows       : {len(patients):,}")
print(f"Unique IDs : {patients['Patient_ID'].nunique():,}")

print("\nPayers:")
print(f"Rows       : {len(payers):,}")
print(f"Unique IDs : {payers['Payer_ID'].nunique():,}")

print("\nProviders:")
print(f"Rows       : {len(providers):,}")
print(f"Unique IDs : {providers['Provider_ID'].nunique():,}")

print("\nProcedures:")
print(f"Rows       : {len(procedures):,}")
print(f"Unique IDs : {procedures['Procedure_Code'].nunique():,}")


# =========================================================
# 6. DISPLAY SAMPLE DATA
# =========================================================

print("\n========== PAYER SAMPLE ==========")
display(payers)

print("\n========== PROVIDER SAMPLE ==========")
display(providers.head(10))

print("\n========== PROCEDURE SAMPLE ==========")
display(procedures)

print("\n========== PATIENT SAMPLE ==========")
display(patients.head(10))

========== MASTER DATA VALIDATION ==========

Patients:
Rows       : 15,000
Unique IDs : 15,000

Payers:
Rows       : 8
Unique IDs : 8

Providers:
Rows       : 40
Unique IDs : 40

Procedures:
Rows       : 24
Unique IDs : 24

========== PAYER SAMPLE ==========


,Payer_ID,Payer_Name,Payer_Type
0,PY001,SecureHealth Insurance,Private
1,PY002,PrimeCare Health,Private
2,PY003,MediSure Insurance,Private
3,PY004,WellLife Health,Private
4,PY005,NationalCare Health,Government
5,PY006,United Medical Assurance,Private
6,PY007,HealthFirst Insurance,Private
7,PY008,CareShield Health,Private



========== PROVIDER SAMPLE ==========


,Provider_ID,Provider_Name,Location,Provider_Type,Specialty_Profile
0,PR001,MedCore Provider 01,Chennai Central,Hospital,Cardiology|General Medicine
1,PR002,MedCore Provider 02,Chennai East,Hospital,Radiology|General Medicine
2,PR003,MedCore Provider 03,Chennai North,Hospital,Orthopedics|Emergency Medicine
3,PR004,MedCore Provider 04,Chennai South,Hospital,Oncology|General Medicine
4,PR005,MedCore Provider 05,Chennai West,Hospital,Neurology|General Medicine
5,PR006,MedCore Provider 06,Coimbatore,Hospital,Gastroenterology|General Medicine
6,PR007,MedCore Provider 07,Chennai Central,Hospital,General Medicine|Emergency Medicine
7,PR008,MedCore Provider 08,Chennai East,Hospital,Cardiology|Radiology|General Medicine
8,PR009,MedCore Provider 09,Chennai North,Hospital,Cardiology|General Medicine
9,PR010,MedCore Provider 10,Chennai South,Hospital,Radiology|General Medicine



========== PROCEDURE SAMPLE ==========


,Procedure_Code,Procedure_Name,Specialty,Authorization_Required,Min_Claim_Amount,Max_Claim_Amount
0,PC001,ECG,Cardiology,False,800,2500
1,PC002,Echocardiogram,Cardiology,False,2500,6000
2,PC003,Cardiac Stress Test,Cardiology,True,3000,7500
3,PC004,Cardiac Catheterization,Cardiology,True,45000,120000
4,PC005,X-Ray,Radiology,False,500,2000
5,PC006,CT Scan,Radiology,True,5000,15000
6,PC007,MRI Brain,Radiology,True,8000,20000
7,PC008,MRI Spine,Radiology,True,9000,22000
8,PC009,Ultrasound,Radiology,False,1000,4000
9,PC010,Fracture Treatment,Orthopedics,False,8000,30000



========== PATIENT SAMPLE ==========


,Patient_ID,Age,Gender
0,PAT00001,24,Female
1,PAT00002,70,Male
2,PAT00003,62,Female
3,PAT00004,47,Female
4,PAT00005,47,Male
5,PAT00006,76,Male
6,PAT00007,23,Male
7,PAT00008,65,Male
8,PAT00009,31,Male
9,PAT00010,24,Male


In [2]:
# =========================================================
# STEP 2 — GENERATE 50,000 CLAIMS
# =========================================================

number_of_claims = 50000

# ---------------------------------------------------------
# 1. Create provider-specialty mapping
# ---------------------------------------------------------

provider_specialties = []

for _, row in providers.iterrows():

    specialties = row["Specialty_Profile"].split("|")

    for specialty in specialties:

        provider_specialties.append({
            "Provider_ID": row["Provider_ID"],
            "Specialty": specialty
        })

provider_specialties = pd.DataFrame(provider_specialties)


# ---------------------------------------------------------
# 2. Create specialty → procedure mapping
# ---------------------------------------------------------

specialty_procedures = {}

for specialty in procedures["Specialty"].unique():

    specialty_procedures[specialty] = procedures[
        procedures["Specialty"] == specialty
    ].copy()


# ---------------------------------------------------------
# 3. Payer selection probabilities
# ---------------------------------------------------------

payer_ids = payers["Payer_ID"].tolist()

payer_probabilities = [
    0.14,  # PY001
    0.16,  # PY002
    0.13,  # PY003
    0.12,  # PY004
    0.10,  # PY005
    0.13,  # PY006
    0.11,  # PY007
    0.11   # PY008
]


# ---------------------------------------------------------
# 4. Generate claims
# ---------------------------------------------------------

claim_rows = []

for i in range(1, number_of_claims + 1):

    # -----------------------------
    # Select provider
    # -----------------------------

    provider_id = rng.choice(
        providers["Provider_ID"]
    )

    provider_record = providers[
        providers["Provider_ID"] == provider_id
    ].iloc[0]

    available_specialties = (
        provider_record["Specialty_Profile"].split("|")
    )

    specialty = rng.choice(
        available_specialties
    )


    # -----------------------------
    # Select procedure
    # -----------------------------

    specialty_procs = specialty_procedures[specialty]

    procedure_record = specialty_procs.sample(
        n=1,
        random_state=int(rng.integers(0, 1_000_000))
    ).iloc[0]


    # -----------------------------
    # Select patient
    # -----------------------------

    patient_id = rng.choice(
        patients["Patient_ID"]
    )


    # -----------------------------
    # Select payer
    # -----------------------------

    payer_id = rng.choice(
        payer_ids,
        p=payer_probabilities
    )


    # -----------------------------
    # Service date
    # -----------------------------

    service_date = pd.Timestamp("2025-01-01") + pd.Timedelta(
        days=int(rng.integers(0, 365))
    )


    # -----------------------------
    # Claim amount
    # -----------------------------

    min_amount = procedure_record["Min_Claim_Amount"]
    max_amount = procedure_record["Max_Claim_Amount"]

    claim_amount = round(
        rng.uniform(min_amount, max_amount),
        2
    )


    # -----------------------------
    # Denial probability
    # -----------------------------

    # Start with a moderate baseline
    denial_probability = 0.07

    # Slight payer differences
    payer_adjustment = {
        "PY001": 0.00,
        "PY002": 0.025,
        "PY003": -0.010,
        "PY004": 0.005,
        "PY005": -0.005,
        "PY006": 0.010,
        "PY007": 0.015,
        "PY008": -0.005
    }

    denial_probability += payer_adjustment[payer_id]


    # Certain authorization-required procedures
    # have somewhat higher denial exposure
    if procedure_record["Authorization_Required"]:
        denial_probability += 0.025


    # Slight specialty variation
    specialty_adjustment = {
        "Cardiology": 0.015,
        "Radiology": 0.010,
        "Orthopedics": 0.005,
        "Oncology": 0.008,
        "Neurology": 0.010,
        "Gastroenterology": 0.005,
        "General Medicine": -0.015,
        "Emergency Medicine": 0.000
    }

    denial_probability += specialty_adjustment[specialty]


    # Keep probabilities within sensible limits
    denial_probability = np.clip(
        denial_probability,
        0.03,
        0.18
    )


    # -----------------------------
    # Decide denial
    # -----------------------------

    denied = rng.random() < denial_probability


    # -----------------------------
    # Claim outcome
    # -----------------------------
    # Denial reasons are assigned AFTER authorization records are created.
    # This lets the reason reflect whether authorization was valid by the
    # service date instead of generating contradictory records independently.
    denial_reason = None

    if denied:
        claim_status = "Denied"
    else:
        # Most non-denied claims are paid, with a small pending portion.
        if rng.random() < 0.94:
            claim_status = "Paid"
        else:
            claim_status = "Pending"


    # -----------------------------
    # Store claim
    # -----------------------------

    claim_rows.append([
        f"CL{i:06d}",
        patient_id,
        provider_id,
        payer_id,
        service_date,
        specialty,
        procedure_record["Procedure_Code"],
        procedure_record["Procedure_Name"],
        claim_amount,
        int(denied),
        denial_reason,
        claim_status
    ])


# ---------------------------------------------------------
# 5. Create Claims DataFrame
# ---------------------------------------------------------

claims = pd.DataFrame(
    claim_rows,
    columns=[
        "Claim_ID",
        "Patient_ID",
        "Provider_ID",
        "Payer_ID",
        "Service_Date",
        "Specialty",
        "Procedure_Code",
        "Procedure_Name",
        "Claim_Amount",
        "Denial_Flag",
        "Denial_Reason",
        "Claim_Status"
    ]
)


# =========================================================
# STEP 2 VALIDATION
# =========================================================

print("========== CLAIM VALIDATION ==========")

print("\nRows:")
print(f"{len(claims):,}")

print("\nUnique Claim IDs:")
print(f"{claims['Claim_ID'].nunique():,}")

print("\nDate range:")
print(claims["Service_Date"].min().date())
print("to")
print(claims["Service_Date"].max().date())

print("\nTotal Claim Value:")
print(f"₹{claims['Claim_Amount'].sum():,.2f}")

print("\nAverage Claim Value:")
print(f"₹{claims['Claim_Amount'].mean():,.2f}")

print("\nMedian Claim Value:")
print(f"₹{claims['Claim_Amount'].median():,.2f}")

print("\nDenial Rate:")
print(
    f"{claims['Denial_Flag'].mean() * 100:.2f}%"
)

print("\nClaim Status:")
display(
    claims["Claim_Status"]
    .value_counts()
    .to_frame("Claim_Count")
)

print("\nDenial Reasons:")
print("Denial reasons will be assigned after authorization records are generated,")
print("so the reason can be checked against authorization timing.")

print("\nClaims by Specialty:")
display(
    claims["Specialty"]
    .value_counts()
    .to_frame("Claim_Count")
)

print("\nFirst 10 Claims:")
display(claims.head(10))

========== CLAIM VALIDATION ==========

Rows:
50,000

Unique Claim IDs:
50,000

Date range:
2025-01-01
to
2025-12-31

Total Claim Value:
₹781,292,204.12

Average Claim Value:
₹15,625.84

Median Claim Value:
₹3,507.12

Denial Rate:
8.27%

Claim Status:


,Claim_Count
Claim_Status,
Paid,43113
Denied,4137
Pending,2750



Denial Reasons:
Denial reasons will be assigned after authorization records are generated,
so the reason can be checked against authorization timing.

Claims by Specialty:


,Claim_Count
Specialty,
General Medicine,20689
Emergency Medicine,6310
Cardiology,5280
Radiology,5097
Orthopedics,3195
Neurology,3187
Oncology,3132
Gastroenterology,3110



First 10 Claims:


,Claim_ID,Patient_ID,Provider_ID,Payer_ID,Service_Date,Specialty,Procedure_Code,Procedure_Name,Claim_Amount,Denial_Flag,Denial_Reason,Claim_Status
0,CL000001,PAT03066,PR033,PY001,2025-09-04,General Medicine,PC021,General Consultation,1076.19,0,None,Paid
1,CL000002,PAT13102,PR030,PY006,2025-09-27,Gastroenterology,PC018,Endoscopy,7434.65,0,None,Paid
2,CL000003,PAT07344,PR030,PY002,2025-02-21,General Medicine,PC021,General Consultation,1203.66,0,None,Paid
3,CL000004,PAT14867,PR026,PY005,2025-12-27,General Medicine,PC022,Lab Panel,1867.72,0,None,Paid
4,CL000005,PAT08768,PR029,PY001,2025-08-14,General Medicine,PC021,General Consultation,1653.12,0,None,Paid
5,CL000006,PAT03507,PR009,PY004,2025-10-21,General Medicine,PC022,Lab Panel,4468.31,0,None,Paid
6,CL000007,PAT03804,PR020,PY007,2025-06-17,General Medicine,PC021,General Consultation,909.64,0,None,Paid
7,CL000008,PAT00440,PR025,PY007,2025-01-18,General Medicine,PC021,General Consultation,1383.42,0,None,Paid
8,CL000009,PAT11550,PR001,PY001,2025-11-09,General Medicine,PC022,Lab Panel,4137.34,0,None,Paid
9,CL000010,PAT10624,PR013,PY004,2025-10-03,Neurology,PC017,Neurology Consultation,2287.48,0,None,Paid


In [3]:
# ============================================================
# STEP 3 — GENERATE AUTHORIZATION DATA (CORRECTED)
# ============================================================

authorization_rows = []

for _, claim in claims.iterrows():

    procedure_code = claim["Procedure_Code"]
    service_date = claim["Service_Date"]

    # Get whether authorization is required
    procedure_info = procedures[
        procedures["Procedure_Code"] == procedure_code
    ].iloc[0]

    authorization_required = procedure_info["Authorization_Required"]

    # Default values
    authorization_status = "Not Required"
    request_date = pd.NaT
    approval_date = pd.NaT
    authorization_id = None

    # --------------------------------------------------------
    # If authorization is required
    # --------------------------------------------------------
    if authorization_required == True:

        # Most claims should have authorization requested
        request_probability = rng.random()

        if request_probability < 0.88:

            authorization_id = f"AUTH{len(authorization_rows) + 1:06d}"

            # Request timing
            timing_probability = rng.random()

            if timing_probability < 0.82:
                # Requested before service
                days_before = rng.integers(1, 15)
                request_date = service_date - pd.Timedelta(
                    days=int(days_before)
                )

            elif timing_probability < 0.94:
                # Requested on service date
                request_date = service_date

            else:
                # Late authorization request
                days_after = rng.integers(1, 5)
                request_date = service_date + pd.Timedelta(
                    days=int(days_after)
                )

            # Approval outcome
            approval_probability = rng.random()

            if approval_probability < 0.86:
                authorization_status = "Approved"

                approval_days = rng.integers(0, 5)

                approval_date = request_date + pd.Timedelta(
                    days=int(approval_days)
                )

            elif approval_probability < 0.94:
                authorization_status = "Pending"

            else:
                authorization_status = "Denied"

        else:
            # No authorization request recorded
            authorization_status = "Not Requested"

    else:
        authorization_status = "Not Required"


    authorization_rows.append({
        "Claim_ID": claim["Claim_ID"],
        "Authorization_Required": authorization_required,
        "Authorization_Status": authorization_status,
        "Request_Date": request_date,
        "Approval_Date": approval_date,
        "Authorization_ID": authorization_id
    })


# Create DataFrame
authorization = pd.DataFrame(authorization_rows)


# ============================================================
# VALIDATION
# ============================================================

print("\n========== AUTHORIZATION VALIDATION ==========")

print("\nRows:")
print(f"{len(authorization):,}")

print("\nUnique Claim IDs:")
print(authorization["Claim_ID"].nunique())

print("\nAuthorization Required:")
print(
    authorization["Authorization_Required"]
    .value_counts()
    .to_frame("Claim_Count")
)

print("\nAuthorization Status:")
print(
    authorization["Authorization_Status"]
    .value_counts()
    .to_frame("Claim_Count")
)

print("\nAuthorization IDs:")
print(
    authorization["Authorization_ID"]
    .notna()
    .sum()
)

print("\nFirst 10 Authorization Records:")
display(authorization.head(10))


========== AUTHORIZATION VALIDATION ==========

Rows:
50,000

Unique Claim IDs:
50000

Authorization Required:
                        Claim_Count
Authorization_Required             
False                         33340
True                          16660

Authorization Status:
                      Claim_Count
Authorization_Status             
Not Required                33340
Approved                    12642
Not Requested                2002
Pending                      1145
Denied                        871

Authorization IDs:
14658

First 10 Authorization Records:


,Claim_ID,Authorization_Required,Authorization_Status,Request_Date,Approval_Date,Authorization_ID
0,CL000001,False,Not Required,NaT,NaT,None
1,CL000002,True,Approved,2025-09-13,2025-09-15,AUTH000002
2,CL000003,False,Not Required,NaT,NaT,None
3,CL000004,False,Not Required,NaT,NaT,None
4,CL000005,False,Not Required,NaT,NaT,None
5,CL000006,False,Not Required,NaT,NaT,None
6,CL000007,False,Not Required,NaT,NaT,None
7,CL000008,False,Not Required,NaT,NaT,None
8,CL000009,False,Not Required,NaT,NaT,None
9,CL000010,False,Not Required,NaT,NaT,None


In [4]:
# ============================================================
# FINALIZE DENIAL REASONS AFTER AUTHORIZATION GENERATION
# ============================================================
# Business rule for this synthetic dataset:
# An authorization approved on or before the service date is considered valid.
# If authorization was approved only after service, Missing Authorization may
# still be a plausible denial reason. Other denial reasons remain possible.

claim_auth = claims[["Claim_ID", "Procedure_Code", "Service_Date", "Denial_Flag"]].merge(
    authorization,
    on="Claim_ID",
    how="left",
    validate="one_to_one"
)

for idx, row in claim_auth.iterrows():
    if row["Denial_Flag"] != 1:
        claims.at[idx, "Denial_Reason"] = None
        continue

    procedure_requires_auth = bool(row["Authorization_Required"])
    approved_by_service = (
        row["Authorization_Status"] == "Approved"
        and pd.notna(row["Approval_Date"])
        and row["Approval_Date"] <= row["Service_Date"]
    )

    if procedure_requires_auth:
        if approved_by_service:
            # Do not assign Missing Authorization when approval was valid in time.
            reasons = ["Eligibility Issue", "Coding Error", "Medical Necessity", "Documentation Missing"]
            weights = [0.15, 0.18, 0.20, 0.15]
        else:
            reasons = ["Missing Authorization", "Eligibility Issue", "Coding Error", "Medical Necessity", "Documentation Missing"]
            weights = [0.32, 0.15, 0.18, 0.20, 0.15]
    else:
        reasons = ["Eligibility Issue", "Coding Error", "Duplicate Claim", "Timely Filing", "Medical Necessity", "Documentation Missing"]
        weights = [0.18, 0.22, 0.08, 0.12, 0.15, 0.25]

    claims.at[idx, "Denial_Reason"] = rng.choice(reasons, p=np.array(weights) / np.sum(weights))

print("Denial reasons finalized using authorization timing.")
print("Denial reasons (denied claims only):")
display(claims.loc[claims["Denial_Flag"] == 1, "Denial_Reason"].value_counts().to_frame("Denied_Claims"))


Denial reasons finalized using authorization timing.
Denial reasons (denied claims only):


,Denied_Claims
Denial_Reason,
Coding Error,946
Documentation Missing,916
Medical Necessity,822
Eligibility Issue,775
Timely Filing,265
Missing Authorization,236
Duplicate Claim,177


In [5]:
# INVESTIGATION — CHECK AUTHORIZATION / DENIAL REASON CONSISTENCY

claim_auth_check = claims.merge(authorization, on="Claim_ID", how="inner")

approved_by_service = (
    claim_auth_check["Authorization_Status"].eq("Approved")
    & claim_auth_check["Approval_Date"].notna()
    & (claim_auth_check["Approval_Date"] <= claim_auth_check["Service_Date"])
)

contradictions = claim_auth_check[
    (claim_auth_check["Denial_Reason"] == "Missing Authorization")
    & approved_by_service
]

print("Denied claims labelled Missing Authorization despite approval on/before service:", len(contradictions))
print("Expected result: 0")
display(contradictions.head(10))


Denied claims labelled Missing Authorization despite approval on/before service: 0
Expected result: 0


,Claim_ID,Patient_ID,Provider_ID,Payer_ID,Service_Date,Specialty,Procedure_Code,Procedure_Name,Claim_Amount,Denial_Flag,Denial_Reason,Claim_Status,Authorization_Required,Authorization_Status,Request_Date,Approval_Date,Authorization_ID


In [6]:
# ============================================================
# STEP 4 — GENERATE PAYMENTS DATA
# ============================================================

payment_rows = []

for _, claim in claims.iterrows():

    claim_id = claim["Claim_ID"]
    claim_amount = claim["Claim_Amount"]
    denial_flag = claim["Denial_Flag"]
    claim_status = claim["Claim_Status"]

    # --------------------------------------------------------
    # NON-DENIED CLAIMS
    # --------------------------------------------------------

    if denial_flag == 0:

        # Paid claims receive most/all of the claim amount
        if claim_status == "Paid":

            payment_rate = rng.uniform(0.88, 1.00)
            paid_amount = round(claim_amount * payment_rate, 2)

            recovery_status = "Collected"

        # Pending claims have no payment yet
        else:

            paid_amount = 0.00
            recovery_status = "Pending"


    # --------------------------------------------------------
    # DENIED CLAIMS
    # --------------------------------------------------------

    else:

        # Some denied claims are eventually recovered
        recovery_probability = rng.random()

        if recovery_probability < 0.60:

            # Recovered claims receive partial or full payment
            payment_rate = rng.uniform(0.50, 0.95)
            paid_amount = round(claim_amount * payment_rate, 2)

            recovery_status = "Recovered"

        else:

            # Unrecovered denial
            paid_amount = 0.00
            recovery_status = "Unrecovered"


    # Calculate outstanding amount
    outstanding_amount = round(
        claim_amount - paid_amount,
        2
    )

    # Payment date
    if paid_amount > 0:

        # Payment occurs after service
        payment_days = rng.integers(7, 61)

        payment_date = (
            claim["Service_Date"]
            + pd.Timedelta(days=int(payment_days))
        )

    else:

        payment_date = pd.NaT


    payment_rows.append({
        "Claim_ID": claim_id,
        "Paid_Amount": paid_amount,
        "Payment_Date": payment_date,
        "Recovery_Status": recovery_status,
        "Outstanding_Amount": outstanding_amount
    })


# Create Payments DataFrame
payments = pd.DataFrame(payment_rows)


# ============================================================
# VALIDATION
# ============================================================

print("\n========== PAYMENTS VALIDATION ==========")

print("\nRows:")
print(f"{len(payments):,}")

print("\nUnique Claim IDs:")
print(payments["Claim_ID"].nunique())

print("\nTotal Claim Amount:")
print(
    f"₹{claims['Claim_Amount'].sum():,.2f}"
)

print("\nTotal Paid Amount:")
print(
    f"₹{payments['Paid_Amount'].sum():,.2f}"
)

print("\nTotal Outstanding Amount:")
print(
    f"₹{payments['Outstanding_Amount'].sum():,.2f}"
)

print("\nRecovery Status:")
print(
    payments["Recovery_Status"]
    .value_counts()
    .to_frame("Claim_Count")
)

print("\nAverage Paid Amount:")
print(
    f"₹{payments['Paid_Amount'].mean():,.2f}"
)

print("\nFirst 10 Payment Records:")
display(payments.head(10))


========== PAYMENTS VALIDATION ==========

Rows:
50,000

Unique Claim IDs:
50000

Total Claim Amount:
₹781,292,204.12

Total Paid Amount:
₹649,622,674.09

Total Outstanding Amount:
₹131,669,530.03

Recovery Status:
                 Claim_Count
Recovery_Status             
Collected              43113
Pending                 2750
Recovered               2513
Unrecovered             1624

Average Paid Amount:
₹12,992.45

First 10 Payment Records:


,Claim_ID,Paid_Amount,Payment_Date,Recovery_Status,Outstanding_Amount
0,CL000001,987.82,2025-09-12,Collected,88.37
1,CL000002,7019.22,2025-10-13,Collected,415.43
2,CL000003,1144.66,2025-03-18,Collected,59.00
3,CL000004,1792.27,2026-02-07,Collected,75.45
4,CL000005,1516.98,2025-10-10,Collected,136.14
5,CL000006,4091.01,2025-11-06,Collected,377.30
6,CL000007,886.27,2025-07-01,Collected,23.37
7,CL000008,1273.91,2025-02-20,Collected,109.51
8,CL000009,4003.95,2025-11-24,Collected,133.39
9,CL000010,2169.96,2025-11-25,Collected,117.52


In [7]:
# ============================================================
# STEP 5 — DATA MODEL AND BUSINESS-RULE VALIDATION
# ============================================================
# This cell intentionally raises an AssertionError if a rule fails.
# CSV export is allowed only after every assertion passes.

checks = []
def check(name, condition):
    passed = bool(condition)
    checks.append((name, passed))
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
    assert passed, f"Validation failed: {name}"

# Master key uniqueness
check("Patient_ID is unique", patients["Patient_ID"].is_unique)
check("Payer_ID is unique", payers["Payer_ID"].is_unique)
check("Provider_ID is unique", providers["Provider_ID"].is_unique)
check("Procedure_Code is unique", procedures["Procedure_Code"].is_unique)

# Expected row counts and claim keys
check("Claims contain expected 50,000 rows", len(claims) == number_of_claims)
check("Claim_ID is unique", claims["Claim_ID"].is_unique)
check("One authorization row per claim", len(authorization) == len(claims) and authorization["Claim_ID"].is_unique)
check("One payment row per claim", len(payments) == len(claims) and payments["Claim_ID"].is_unique)

# Foreign keys
check("All Patient_ID values match patient master", claims["Patient_ID"].isin(patients["Patient_ID"]).all())
check("All Provider_ID values match provider master", claims["Provider_ID"].isin(providers["Provider_ID"]).all())
check("All Payer_ID values match payer master", claims["Payer_ID"].isin(payers["Payer_ID"]).all())
check("All Procedure_Code values match procedure master", claims["Procedure_Code"].isin(procedures["Procedure_Code"]).all())
check("All authorization Claim_ID values match claims", authorization["Claim_ID"].isin(claims["Claim_ID"]).all())
check("All payment Claim_ID values match claims", payments["Claim_ID"].isin(claims["Claim_ID"]).all())

# Claim outcome consistency
check("Denial_Flag is 1 exactly for Denied claims", ((claims["Denial_Flag"] == 1) == (claims["Claim_Status"] == "Denied")).all())
check("Denied claims have a denial reason", claims.loc[claims["Claim_Status"] == "Denied", "Denial_Reason"].notna().all())
check("Paid/Pending claims have no denial reason", claims.loc[claims["Claim_Status"] != "Denied", "Denial_Reason"].isna().all())

# Authorization requirement must match procedure master
claim_procedure = claims[["Claim_ID", "Procedure_Code", "Service_Date", "Denial_Reason"]].merge(
    procedures[["Procedure_Code", "Authorization_Required"]], on="Procedure_Code", how="left", validate="many_to_one"
)
auth_check = authorization.merge(claim_procedure, on="Claim_ID", how="left", suffixes=("_auth", "_procedure"), validate="one_to_one")
check("Authorization_Required matches procedure master", (auth_check["Authorization_Required_auth"] == auth_check["Authorization_Required_procedure"]).all())
check("Non-required authorization records say Not Required", (auth_check.loc[~auth_check["Authorization_Required_auth"], "Authorization_Status"] == "Not Required").all())
check("Non-required authorization records have no auth dates or ID", auth_check.loc[~auth_check["Authorization_Required_auth"], ["Request_Date", "Approval_Date", "Authorization_ID"]].isna().all().all())
check("Approved records have request and approval dates", auth_check.loc[auth_check["Authorization_Status"] == "Approved", ["Request_Date", "Approval_Date"]].notna().all().all())
check("Approval date is not before request date", (auth_check.loc[auth_check["Approval_Date"].notna(), "Approval_Date"] >= auth_check.loc[auth_check["Approval_Date"].notna(), "Request_Date"]).all())

approved_by_service = (
    auth_check["Authorization_Status"].eq("Approved")
    & auth_check["Approval_Date"].notna()
    & (auth_check["Approval_Date"] <= auth_check["Service_Date"])
)
check("No Missing Authorization denial when approved on/before service", ~((auth_check["Denial_Reason"] == "Missing Authorization") & approved_by_service).any())

# Payment/recovery logic
payment_check = payments.merge(claims[["Claim_ID", "Claim_Status", "Claim_Amount"]], on="Claim_ID", how="left", validate="one_to_one")
check("Paid_Amount is never negative", (payment_check["Paid_Amount"] >= 0).all())
check("Paid_Amount never exceeds Claim_Amount", (payment_check["Paid_Amount"] <= payment_check["Claim_Amount"] + 0.011).all())
check("Outstanding_Amount equals claim amount minus paid amount", np.allclose(payment_check["Outstanding_Amount"], (payment_check["Claim_Amount"] - payment_check["Paid_Amount"]).round(2), atol=0.01))
check("Positive payments have a payment date", payment_check.loc[payment_check["Paid_Amount"] > 0, "Payment_Date"].notna().all())
check("Zero payments have no payment date", payment_check.loc[payment_check["Paid_Amount"] == 0, "Payment_Date"].isna().all())
check("Paid claims have Collected recovery status", (payment_check.loc[payment_check["Claim_Status"] == "Paid", "Recovery_Status"] == "Collected").all())
check("Pending claims have Pending recovery status and zero paid amount", ((payment_check.loc[payment_check["Claim_Status"] == "Pending", "Recovery_Status"] == "Pending").all() and (payment_check.loc[payment_check["Claim_Status"] == "Pending", "Paid_Amount"] == 0).all()))
check("Denied claims are marked Recovered or Unrecovered", payment_check.loc[payment_check["Claim_Status"] == "Denied", "Recovery_Status"].isin(["Recovered", "Unrecovered"]).all())

print(f"\nAll {len(checks)} validation checks passed. Safe to export to the corrected output folder.")


PASS — Patient_ID is unique
PASS — Payer_ID is unique
PASS — Provider_ID is unique
PASS — Procedure_Code is unique
PASS — Claims contain expected 50,000 rows
PASS — Claim_ID is unique
PASS — One authorization row per claim
PASS — One payment row per claim
PASS — All Patient_ID values match patient master
PASS — All Provider_ID values match provider master
PASS — All Payer_ID values match payer master
PASS — All Procedure_Code values match procedure master
PASS — All authorization Claim_ID values match claims
PASS — All payment Claim_ID values match claims
PASS — Denial_Flag is 1 exactly for Denied claims
PASS — Denied claims have a denial reason
PASS — Paid/Pending claims have no denial reason
PASS — Authorization_Required matches procedure master
PASS — Non-required authorization records say Not Required
PASS — Non-required authorization records have no auth dates or ID
PASS — Approved records have request and approval dates
PASS — Approval date is not before request date
PASS — No Mi

In [8]:
# ============================================================
# STEP 6 — EXPORT VALIDATED TABLES TO A NEW FOLDER
# ============================================================
# This corrected notebook writes to a separate folder and does not overwrite
# the existing MedCore_RCM_Data outputs or any source notebook.

from pathlib import Path

output_folder = Path("MedCore_RCM_Data_Corrected")
output_folder.mkdir(parents=True, exist_ok=True)

tables_to_export = {
    "Patients.csv": patients,
    "Payers.csv": payers,
    "Providers.csv": providers,
    "Procedures.csv": procedures,
    "Claims.csv": claims,
    "Authorization.csv": authorization,
    "Payments.csv": payments,
}

# Prevent accidental overwrite of files already in the corrected folder.
existing_files = [name for name in tables_to_export if (output_folder / name).exists()]
if existing_files:
    raise FileExistsError(
        f"Export stopped to protect existing files in {output_folder}: {existing_files}. "
        "Rename or move that folder before rerunning if you intentionally want a fresh export."
    )

for file_name, dataframe in tables_to_export.items():
    dataframe.to_csv(output_folder / file_name, index=False)

print("========== CSV EXPORT COMPLETE ==========")
print(f"Output folder: {output_folder.resolve()}")
for file_name in tables_to_export:
    print(f"✓ {file_name}")


========== CSV EXPORT COMPLETE ==========
Output folder: C:\Users\admin\Anaconda Project\Project 3 New\MedCore_RCM_Data\MedCore_RCM_Data_Corrected
✓ Patients.csv
✓ Payers.csv
✓ Providers.csv
✓ Procedures.csv
✓ Claims.csv
✓ Authorization.csv
✓ Payments.csv


In [1]:

# ================================================================
# MEDCORE RCM — FINAL BUSINESS RULE & DATA QUALITY AUDIT
# ================================================================
# READ-ONLY AUDIT:
# - Does not change any existing DataFrame.
# - Does not regenerate or export data.
# - Separates hard errors from review flags.
#
# Run after these DataFrames exist:
# patients, payers, providers, procedures,
# claims, authorization, payments
# ================================================================

import pandas as pd
import numpy as np
from IPython.display import display

hard_errors = []
review_flags = []

def add_findings(bucket, rule, rows):
    """Store only non-empty finding tables."""
    if len(rows) > 0:
        bucket.append({
            "Rule": rule,
            "Count": len(rows),
            "Details": rows.copy()
        })

def add_count_finding(bucket, rule, expected, actual):
    if expected != actual:
        add_findings(
            bucket,
            rule,
            pd.DataFrame([{
                "Expected": expected,
                "Actual": actual
            }])
        )

# ------------------------------------------------
# 1. WORK ON COPIES — ORIGINAL DATA IS UNCHANGED
# ------------------------------------------------

p = patients.copy()
py = payers.copy()
pr = providers.copy()
proc = procedures.copy()
c = claims.copy()
a = authorization.copy()
pay = payments.copy()

date_columns = {
    "claims": (c, ["Service_Date"]),
    "authorization": (a, ["Request_Date", "Approval_Date"]),
    "payments": (pay, ["Payment_Date"])
}

for df, columns in date_columns.values():
    for col in columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Numeric conversions on copies
c["Claim_Amount"] = pd.to_numeric(c["Claim_Amount"], errors="coerce")
pay["Paid_Amount"] = pd.to_numeric(pay["Paid_Amount"], errors="coerce")
pay["Outstanding_Amount"] = pd.to_numeric(
    pay["Outstanding_Amount"], errors="coerce"
)

# ------------------------------------------------
# 2. EXPECTED TABLE SIZES
# ------------------------------------------------

expected_rows = {
    "Patients": 15000,
    "Payers": 8,
    "Providers": 40,
    "Procedures": 24,
    "Claims": 50000,
    "Authorization": 50000,
    "Payments": 50000
}

actual_rows = {
    "Patients": len(p),
    "Payers": len(py),
    "Providers": len(pr),
    "Procedures": len(proc),
    "Claims": len(c),
    "Authorization": len(a),
    "Payments": len(pay)
}

for table, expected in expected_rows.items():
    add_count_finding(
        hard_errors,
        f"{table}: unexpected row count",
        expected,
        actual_rows[table]
    )

# ------------------------------------------------
# 3. PRIMARY KEY AND DUPLICATE CHECKS
# ------------------------------------------------

primary_keys = {
    "Patients": (p, "Patient_ID"),
    "Payers": (py, "Payer_ID"),
    "Providers": (pr, "Provider_ID"),
    "Procedures": (proc, "Procedure_Code"),
    "Claims": (c, "Claim_ID"),
    "Authorization": (a, "Claim_ID"),
    "Payments": (pay, "Claim_ID")
}

for table, (df, key) in primary_keys.items():

    duplicate_ids = df[df.duplicated(key, keep=False)]

    add_findings(
        hard_errors,
        f"{table}: duplicate {key}",
        duplicate_ids
    )

    add_findings(
        hard_errors,
        f"{table}: completely duplicated rows",
        df[df.duplicated(keep=False)]
    )

# Authorization IDs should also be unique when present
auth_id_duplicates = a[
    a["Authorization_ID"].notna()
    & a.duplicated("Authorization_ID", keep=False)
]

add_findings(
    hard_errors,
    "Duplicate non-null Authorization_ID values",
    auth_id_duplicates
)

# ------------------------------------------------
# 4. REQUIRED FIELD / MISSING VALUE CHECKS
# ------------------------------------------------

master_tables = {
    "Patients": p,
    "Payers": py,
    "Providers": pr,
    "Procedures": proc
}

for table, df in master_tables.items():
    missing = df[df.isna().any(axis=1)]

    add_findings(
        hard_errors,
        f"{table}: missing values in master data",
        missing
    )

required_claim_columns = [
    "Claim_ID", "Patient_ID", "Provider_ID", "Payer_ID",
    "Service_Date", "Specialty", "Procedure_Code",
    "Procedure_Name", "Claim_Amount", "Denial_Flag",
    "Claim_Status"
]

add_findings(
    hard_errors,
    "Claims: missing required fields",
    c[c[required_claim_columns].isna().any(axis=1)]
)

add_findings(
    hard_errors,
    "Authorization: missing required fields",
    a[a[["Claim_ID", "Authorization_Required",
         "Authorization_Status"]].isna().any(axis=1)]
)

add_findings(
    hard_errors,
    "Payments: missing required non-date fields",
    pay[pay[[
        "Claim_ID", "Paid_Amount",
        "Recovery_Status", "Outstanding_Amount"
    ]].isna().any(axis=1)]
)

# ------------------------------------------------
# 5. FOREIGN KEY / TABLE RELATIONSHIP CHECKS
# ------------------------------------------------

foreign_key_checks = [
    ("Claims: invalid Patient_ID",
     c[~c["Patient_ID"].isin(p["Patient_ID"])]),

    ("Claims: invalid Provider_ID",
     c[~c["Provider_ID"].isin(pr["Provider_ID"])]),

    ("Claims: invalid Payer_ID",
     c[~c["Payer_ID"].isin(py["Payer_ID"])]),

    ("Claims: invalid Procedure_Code",
     c[~c["Procedure_Code"].isin(proc["Procedure_Code"])]),

    ("Authorization: Claim_ID not found in Claims",
     a[~a["Claim_ID"].isin(c["Claim_ID"])]),

    ("Payments: Claim_ID not found in Claims",
     pay[~pay["Claim_ID"].isin(c["Claim_ID"])])
]

for rule, rows in foreign_key_checks:
    add_findings(hard_errors, rule, rows)

# Check that each claim has exactly one authorization and payment record
missing_auth = c[~c["Claim_ID"].isin(a["Claim_ID"])]
missing_payment = c[~c["Claim_ID"].isin(pay["Claim_ID"])]

add_findings(
    hard_errors,
    "Claims without authorization records",
    missing_auth
)

add_findings(
    hard_errors,
    "Claims without payment records",
    missing_payment
)

# ------------------------------------------------
# 6. CLAIM STATUS AND DENIAL REASON CHECKS
# ------------------------------------------------

valid_claim_statuses = {"Paid", "Denied", "Pending"}

add_findings(
    hard_errors,
    "Invalid Claim_Status values",
    c[~c["Claim_Status"].isin(valid_claim_statuses)]
)

add_findings(
    hard_errors,
    "Invalid Denial_Flag values",
    c[~c["Denial_Flag"].isin([0, 1])]
)

# Denial flag must agree with claim status
bad_flag = c[
    (c["Denial_Flag"] == 1) !=
    (c["Claim_Status"] == "Denied")
]

add_findings(
    hard_errors,
    "Denial_Flag and Claim_Status do not agree",
    bad_flag
)

# Denied claims must have a reason
add_findings(
    hard_errors,
    "Denied claims without a Denial_Reason",
    c[
        (c["Claim_Status"] == "Denied")
        & c["Denial_Reason"].isna()
    ]
)

# Non-denied claims should not have a denial reason
add_findings(
    hard_errors,
    "Paid/Pending claims with a Denial_Reason",
    c[
        (c["Claim_Status"] != "Denied")
        & c["Denial_Reason"].notna()
    ]
)

# ------------------------------------------------
# 7. PROCEDURE, SPECIALTY AND PROVIDER CONSISTENCY
# ------------------------------------------------

claim_proc = c.merge(
    proc[[
        "Procedure_Code", "Procedure_Name", "Specialty",
        "Authorization_Required", "Min_Claim_Amount",
        "Max_Claim_Amount"
    ]],
    on="Procedure_Code",
    how="left",
    suffixes=("_claim", "_master"),
    validate="many_to_one"
)

# Claim procedure name must match the master
add_findings(
    hard_errors,
    "Claim Procedure_Name does not match procedure master",
    claim_proc[
        claim_proc["Procedure_Name_claim"]
        != claim_proc["Procedure_Name_master"]
    ]
)

# Claim specialty must match procedure specialty
add_findings(
    hard_errors,
    "Claim Specialty does not match procedure specialty",
    claim_proc[
        claim_proc["Specialty_claim"]
        != claim_proc["Specialty_master"]
    ]
)

# Claim specialty must be included in provider profile
provider_profile = pr.set_index("Provider_ID")["Specialty_Profile"].to_dict()

provider_specialty_check = c[
    ["Claim_ID", "Provider_ID", "Specialty"]
].copy()

provider_specialty_check["Provider_Profile"] = (
    provider_specialty_check["Provider_ID"].map(provider_profile)
)

bad_provider_specialty = provider_specialty_check[
    provider_specialty_check.apply(
        lambda row: (
            pd.isna(row["Provider_Profile"])
            or row["Specialty"] not in str(row["Provider_Profile"]).split("|")
        ),
        axis=1
    )
]

add_findings(
    hard_errors,
    "Claim Specialty not included in provider's specialty profile",
    bad_provider_specialty
)

# ------------------------------------------------
# 8. CLAIM AMOUNT AND DATE CHECKS
# ------------------------------------------------

add_findings(
    hard_errors,
    "Zero or negative Claim_Amount",
    c[c["Claim_Amount"] <= 0]
)

add_findings(
    hard_errors,
    "Claim amounts outside procedure master limits",
    claim_proc[
        (claim_proc["Claim_Amount"] < claim_proc["Min_Claim_Amount"] - 0.01)
        | (claim_proc["Claim_Amount"] > claim_proc["Max_Claim_Amount"] + 0.01)
    ]
)

add_findings(
    hard_errors,
    "Service dates outside the generated year 2025",
    c[
        c["Service_Date"].isna()
        | (c["Service_Date"].dt.year != 2025)
    ]
)

# Patient demographic checks
add_findings(
    hard_errors,
    "Patient age outside generated range 18–85",
    p[(p["Age"] < 18) | (p["Age"] > 85)]
)

add_findings(
    hard_errors,
    "Unexpected Gender values",
    p[~p["Gender"].isin(["Female", "Male"])]
)

# ------------------------------------------------
# 9. AUTHORIZATION BUSINESS RULE CHECKS
# ------------------------------------------------

ca = c[[
    "Claim_ID",
    "Service_Date",
    "Denial_Flag",
    "Denial_Reason",
    "Procedure_Code",
    "Claim_Status"          # Added missing column
]].merge(
    a,
    on="Claim_ID",
    how="left",
    validate="one_to_one"
)

ca = ca.merge(
    proc[["Procedure_Code", "Authorization_Required"]].rename(
        columns={"Authorization_Required": "Procedure_Auth_Required"}
    ),
    on="Procedure_Code",
    how="left",
    validate="many_to_one"
)

# Authorization requirement must match the procedure master
add_findings(
    hard_errors,
    "Authorization_Required differs from procedure master",
    ca[
        ca["Authorization_Required"]
        != ca["Procedure_Auth_Required"]
    ]
)

valid_auth_statuses = {
    "Approved", "Pending", "Denied",
    "Not Requested", "Not Required"
}

add_findings(
    hard_errors,
    "Invalid Authorization_Status values",
    ca[~ca["Authorization_Status"].isin(valid_auth_statuses)]
)

# Non-required procedures must have Not Required
add_findings(
    hard_errors,
    "Non-required procedures not marked Not Required",
    ca[
        (~ca["Authorization_Required"])
        & (ca["Authorization_Status"] != "Not Required")
    ]
)

# Required procedures must not say Not Required
add_findings(
    hard_errors,
    "Required procedures marked Not Required",
    ca[
        ca["Authorization_Required"]
        & (ca["Authorization_Status"] == "Not Required")
    ]
)

# Not Required must not have dates or authorization ID
add_findings(
    hard_errors,
    "Not Required records containing authorization details",
    ca[
        (ca["Authorization_Status"] == "Not Required")
        & ca[["Request_Date", "Approval_Date",
              "Authorization_ID"]].notna().any(axis=1)
    ]
)

# Approved records need request date, approval date and ID
add_findings(
    hard_errors,
    "Approved authorization missing date or ID",
    ca[
        (ca["Authorization_Status"] == "Approved")
        & ca[["Request_Date", "Approval_Date",
              "Authorization_ID"]].isna().any(axis=1)
    ]
)

# Pending/Denied requests should have request date and ID,
# but should not have an approval date.
for status in ["Pending", "Denied"]:
    add_findings(
        hard_errors,
        f"{status} authorization missing request date or ID",
        ca[
            (ca["Authorization_Status"] == status)
            & (
                ca["Request_Date"].isna()
                | ca["Authorization_ID"].isna()
            )
        ]
    )

    add_findings(
        hard_errors,
        f"{status} authorization unexpectedly has Approval_Date",
        ca[
            (ca["Authorization_Status"] == status)
            & ca["Approval_Date"].notna()
        ]
    )

# Not Requested must have no request details
add_findings(
    hard_errors,
    "Not Requested records containing request/approval details",
    ca[
        (ca["Authorization_Status"] == "Not Requested")
        & ca[["Request_Date", "Approval_Date",
              "Authorization_ID"]].notna().any(axis=1)
    ]
)

# Approval cannot precede request
add_findings(
    hard_errors,
    "Approval_Date earlier than Request_Date",
    ca[
        ca["Approval_Date"].notna()
        & ca["Request_Date"].notna()
        & (ca["Approval_Date"] < ca["Request_Date"])
    ]
)

# Check request timing against the generation rules:
# 14 days before service to 4 days after service.
ca["Request_Days_From_Service"] = (
    ca["Request_Date"] - ca["Service_Date"]
).dt.days

add_findings(
    hard_errors,
    "Request_Date outside generated timing range (-14 to +4 days)",
    ca[
        ca["Request_Date"].notna()
        & (
            (ca["Request_Days_From_Service"] < -14)
            | (ca["Request_Days_From_Service"] > 4)
        )
    ]
)

# Approval is generated 0–4 days after request
ca["Approval_Days_After_Request"] = (
    ca["Approval_Date"] - ca["Request_Date"]
).dt.days

add_findings(
    hard_errors,
    "Approval delay outside generated range (0–4 days)",
    ca[
        ca["Approval_Date"].notna()
        & ca["Request_Date"].notna()
        & (
            (ca["Approval_Days_After_Request"] < 0)
            | (ca["Approval_Days_After_Request"] > 4)
        )
    ]
)

# Approval valid by service date
approved_by_service = (
    ca["Authorization_Status"].eq("Approved")
    & ca["Approval_Date"].notna()
    & (ca["Approval_Date"] <= ca["Service_Date"])
)

# Missing Authorization is allowed only if:
# authorization is required AND it was not approved by service date.
add_findings(
    hard_errors,
    "Missing Authorization denial despite valid approval by service date",
    ca[
        (ca["Denial_Reason"] == "Missing Authorization")
        & (
            (~ca["Authorization_Required"])
            | approved_by_service
        )
    ]
)

# Missing Authorization cannot apply to procedures that do not require it
add_findings(
    hard_errors,
    "Missing Authorization denial for a non-required procedure",
    ca[
        (ca["Denial_Reason"] == "Missing Authorization")
        & (~ca["Authorization_Required"])
    ]
)

# ------------------------------------------------
# 10. DENIAL REASON ELIGIBILITY CHECK
# ------------------------------------------------

required_auth_reasons = {
    "Missing Authorization",
    "Eligibility Issue",
    "Coding Error",
    "Medical Necessity",
    "Documentation Missing"
}

non_required_auth_reasons = {
    "Eligibility Issue",
    "Coding Error",
    "Duplicate Claim",
    "Timely Filing",
    "Medical Necessity",
    "Documentation Missing"
}

denied_ca = ca[ca["Claim_Status"] == "Denied"].copy()

invalid_reason_rows = denied_ca[
    denied_ca.apply(
        lambda row: (
            row["Denial_Reason"] not in (
                required_auth_reasons
                if row["Authorization_Required"]
                else non_required_auth_reasons
            )
        ),
        axis=1
    )
]

add_findings(
    hard_errors,
    "Denial reason not allowed for authorization requirement type",
    invalid_reason_rows
)

# ------------------------------------------------
# 11. PAYMENT / FINANCIAL CONSISTENCY
# ------------------------------------------------

payment_check = pay.merge(
    c[["Claim_ID", "Claim_Status", "Claim_Amount", "Service_Date"]],
    on="Claim_ID",
    how="left",
    validate="one_to_one"
)

add_findings(
    hard_errors,
    "Negative Paid_Amount",
    payment_check[payment_check["Paid_Amount"] < 0]
)

add_findings(
    hard_errors,
    "Paid_Amount greater than Claim_Amount",
    payment_check[
        payment_check["Paid_Amount"]
        > payment_check["Claim_Amount"] + 0.01
    ]
)

expected_outstanding = (
    payment_check["Claim_Amount"]
    - payment_check["Paid_Amount"]
).round(2)

add_findings(
    hard_errors,
    "Outstanding_Amount does not equal Claim_Amount minus Paid_Amount",
    payment_check[
        ~np.isclose(
            payment_check["Outstanding_Amount"],
            expected_outstanding,
            atol=0.01,
            equal_nan=False
        )
    ]
)

# Payment date rules
add_findings(
    hard_errors,
    "Positive payment without Payment_Date",
    payment_check[
        (payment_check["Paid_Amount"] > 0)
        & payment_check["Payment_Date"].isna()
    ]
)

add_findings(
    hard_errors,
    "Zero payment with a Payment_Date",
    payment_check[
        (payment_check["Paid_Amount"] == 0)
        & payment_check["Payment_Date"].notna()
    ]
)

payment_check["Payment_Days_After_Service"] = (
    payment_check["Payment_Date"]
    - payment_check["Service_Date"]
).dt.days

add_findings(
    hard_errors,
    "Payment_Date outside generated range (7–60 days after service)",
    payment_check[
        (payment_check["Paid_Amount"] > 0)
        & (
            (payment_check["Payment_Days_After_Service"] < 7)
            | (payment_check["Payment_Days_After_Service"] > 60)
        )
    ]
)

# Allowed recovery statuses
valid_recovery_statuses = {
    "Collected", "Pending", "Recovered", "Unrecovered"
}

add_findings(
    hard_errors,
    "Invalid Recovery_Status values",
    payment_check[
        ~payment_check["Recovery_Status"].isin(valid_recovery_statuses)
    ]
)

# Paid claims: Collected and positive payment
add_findings(
    hard_errors,
    "Paid claims not marked Collected or having zero payment",
    payment_check[
        (payment_check["Claim_Status"] == "Paid")
        & (
            (payment_check["Recovery_Status"] != "Collected")
            | (payment_check["Paid_Amount"] <= 0)
        )
    ]
)

# Pending claims: no payment and no payment date
add_findings(
    hard_errors,
    "Pending claims with payment or wrong recovery status",
    payment_check[
        (payment_check["Claim_Status"] == "Pending")
        & (
            (payment_check["Recovery_Status"] != "Pending")
            | (payment_check["Paid_Amount"] != 0)
            | payment_check["Payment_Date"].notna()
        )
    ]
)

# Denied claims: Recovered or Unrecovered
add_findings(
    hard_errors,
    "Denied claims with invalid recovery status",
    payment_check[
        (payment_check["Claim_Status"] == "Denied")
        & (
            ~payment_check["Recovery_Status"].isin(
                ["Recovered", "Unrecovered"]
            )
        )
    ]
)

# Recovered should have a positive payment
add_findings(
    hard_errors,
    "Recovered claims with zero payment",
    payment_check[
        (payment_check["Recovery_Status"] == "Recovered")
        & (payment_check["Paid_Amount"] <= 0)
    ]
)

# Unrecovered should have zero payment
add_findings(
    hard_errors,
    "Unrecovered claims with positive payment",
    payment_check[
        (payment_check["Recovery_Status"] == "Unrecovered")
        & (payment_check["Paid_Amount"] > 0)
    ]
)

# ------------------------------------------------
# 12. PAYMENT LABELS THAT DESERVE REVIEW
# ------------------------------------------------

# Current generator uses 88–100% payment for Paid claims.
# "Collected" with a remaining balance may be better described
# as "Partially Collected" if that is the intended business meaning.

collected_with_balance = payment_check[
    (payment_check["Recovery_Status"] == "Collected")
    & (payment_check["Outstanding_Amount"] > 0.01)
]

add_findings(
    review_flags,
    "Collected status but outstanding balance remains",
    collected_with_balance[[
        "Claim_ID", "Claim_Status", "Claim_Amount",
        "Paid_Amount", "Outstanding_Amount", "Recovery_Status"
    ]]
)

# Recovered denied claims can be partially recovered in this generator.
partial_recoveries = payment_check[
    (payment_check["Recovery_Status"] == "Recovered")
    & (payment_check["Outstanding_Amount"] > 0.01)
]

add_findings(
    review_flags,
    "Recovered status with remaining outstanding balance",
    partial_recoveries[[
        "Claim_ID", "Claim_Amount", "Paid_Amount",
        "Outstanding_Amount", "Recovery_Status"
    ]]
)

# ------------------------------------------------
# 13. UNUSUAL BUT NOT AUTOMATICALLY INVALID PATTERNS
# ------------------------------------------------

# Claims per payer
payer_volume = (
    c.groupby("Payer_ID")
    .agg(Total_Claims=("Claim_ID", "count"))
    .reset_index()
    .merge(py[["Payer_ID", "Payer_Name"]], on="Payer_ID")
    .sort_values("Total_Claims", ascending=False)
)

# Denial rates by payer
payer_denials = (
    c.assign(Denied=(c["Claim_Status"] == "Denied").astype(int))
    .groupby("Payer_ID")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Denied_Claims=("Denied", "sum")
    )
    .reset_index()
)

payer_denials["Denial_Rate_Pct"] = (
    payer_denials["Denied_Claims"]
    / payer_denials["Total_Claims"] * 100
).round(2)

payer_denials = payer_denials.merge(
    py[["Payer_ID", "Payer_Name"]],
    on="Payer_ID"
).sort_values("Denial_Rate_Pct", ascending=False)

# Denial rates by specialty
specialty_denials = (
    c.assign(Denied=(c["Claim_Status"] == "Denied").astype(int))
    .groupby("Specialty")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Denied_Claims=("Denied", "sum"),
        Total_Claim_Amount=("Claim_Amount", "sum")
    )
    .reset_index()
)

specialty_denials["Denial_Rate_Pct"] = (
    specialty_denials["Denied_Claims"]
    / specialty_denials["Total_Claims"] * 100
).round(2)

# Claim amount distribution by procedure
procedure_amounts = (
    c.groupby(["Procedure_Code", "Procedure_Name"])
    .agg(
        Claim_Count=("Claim_ID", "count"),
        Min_Amount=("Claim_Amount", "min"),
        Median_Amount=("Claim_Amount", "median"),
        Average_Amount=("Claim_Amount", "mean"),
        P95_Amount=("Claim_Amount", lambda x: x.quantile(0.95)),
        Max_Amount=("Claim_Amount", "max")
    )
    .reset_index()
)

for col in [
    "Min_Amount", "Median_Amount", "Average_Amount",
    "P95_Amount", "Max_Amount"
]:
    procedure_amounts[col] = procedure_amounts[col].round(2)

# Authorization status vs denial reason
authorization_denial_mix = pd.crosstab(
    ca["Authorization_Status"],
    ca["Denial_Reason"],
    dropna=False
)

# Claim status vs recovery status
payment_status_mix = pd.crosstab(
    payment_check["Claim_Status"],
    payment_check["Recovery_Status"],
    dropna=False
)

# ------------------------------------------------
# 14. FINAL AUDIT SUMMARY
# ------------------------------------------------

hard_summary = pd.DataFrame([
    {"Check": item["Rule"], "Issue_Count": item["Count"]}
    for item in hard_errors
])

review_summary = pd.DataFrame([
    {"Review_Item": item["Rule"], "Record_Count": item["Count"]}
    for item in review_flags
])

total_hard_errors = sum(item["Count"] for item in hard_errors)
total_review_flags = sum(item["Count"] for item in review_flags)

print("\n" + "=" * 70)
print("MEDCORE RCM — FINAL DATA QUALITY AUDIT")
print("=" * 70)

print(f"\nTables checked: 7")
print(f"Total records checked: {sum(actual_rows.values()):,}")
print(f"Hard error groups: {len(hard_errors)}")
print(f"Hard error records: {total_hard_errors:,}")
print(f"Review flag groups: {len(review_flags)}")
print(f"Review flag records: {total_review_flags:,}")

if total_hard_errors == 0:
    print("\nRESULT: PASS — No hard business-rule errors detected.")
else:
    print("\nRESULT: ACTION REQUIRED — Hard errors were detected.")

if total_review_flags > 0:
    print("\nSome records require interpretation or documentation.")
else:
    print("\nNo review flags detected.")

# ------------------------------------------------
# 15. DISPLAY HARD ERRORS
# ------------------------------------------------

if hard_errors:
    print("\n========== HARD ERRORS TO INVESTIGATE ==========")

    display(hard_summary)

    for item in hard_errors:
        print(
            f"\n{item['Rule']} — {item['Count']} record(s)"
        )
        display(item["Details"].head(20))
else:
    print("\nAll hard-error checks passed.")

# ------------------------------------------------
# 16. DISPLAY REVIEW FLAGS
# ------------------------------------------------

if review_flags:
    print("\n========== REVIEW FLAGS — NOT AUTOMATIC ERRORS ==========")

    display(review_summary)

    for item in review_flags:
        print(
            f"\n{item['Rule']} — {item['Count']} record(s)"
        )
        display(item["Details"].head(20))
else:
    print("\nNo review flags.")

# ------------------------------------------------
# 17. BUSINESS DISTRIBUTION REPORTS
# ------------------------------------------------

print("\n========== CLAIM VOLUME BY PAYER ==========")
display(payer_volume)

print("\n========== DENIAL RATE BY PAYER ==========")
display(payer_denials)

print("\n========== DENIAL RATE BY SPECIALTY ==========")
display(
    specialty_denials.sort_values(
        "Denial_Rate_Pct", ascending=False
    )
)

print("\n========== CLAIM AMOUNT DISTRIBUTION BY PROCEDURE ==========")
display(procedure_amounts)

print("\n========== AUTHORIZATION STATUS vs DENIAL REASON ==========")
display(authorization_denial_mix)

print("\n========== CLAIM STATUS vs RECOVERY STATUS ==========")
display(payment_status_mix)

print("\n========== AUDIT COMPLETE ==========")
print("This cell did not modify or export any source data.")


NameError: name 'patients' is not defined